In [ ]:
import heapq #implementa una cola de prioridad

In [ ]:
class HanoiState: # Representa un estado del problema de las Torres de Hanoi
    def __init__(self, rods, num_disks): #Recibe la posición actual de los discos en las varillas y la cantidad total de discos.
        self.rods = tuple(tuple(rod) for rod in rods) # Representamos las varillas como tuplas inmutables de enteros
        self.num_disks = num_disks # Guardamos la cantidad total de discos

    def is_goal(self, target_peg=2): # Verifica si el estado actual es un estado objetivo, es decir, si todos los discos están en la varilla objetivo (por defecto, la varilla 2).
        return len(self.rods[target_peg]) == self.num_disks # Comprueba si la varilla objetivo tiene todos los discos.

    def get_valid_actions(self): # Devuelve una lista de todas las acciones válidas desde el estado actual.
        """Devuelve los movimientos válidos: (varilla_origen, varilla_destino)."""
        actions = [] # Lista de acciones válidas
        for i in range(3): # Itera sobre cada varilla
            if not self.rods[i]: # Si la varilla está vacía, no se puede mover ningún disco desde ella, así que se salta a la siguiente varilla.
                continue
            disk_to_move = self.rods[i][-1] # Obtiene el disco superior de la varilla actual
            for j in range(3): # Itera sobre cada varilla para intentar mover el disco a otra varilla
                if i != j: # No se puede mover un disco a la misma varilla de origen
                    if not self.rods[j] or self.rods[j][-1] > disk_to_move: # Se puede mover el disco si la varilla de destino está vacía o si el disco superior de la varilla de destino es más grande que el disco que se quiere mover.
                        actions.append((i, j)) # Agrega la acción válida (varilla_origen, varilla_destino) a la lista de acciones.
        return actions

    def result(self, action): # Genera un nuevo estado tras aplicar la acción.
        """Genera un nuevo estado tras aplicar la acción."""
        from_peg, to_peg = action # Desempaqueta la acción en varilla de origen y varilla de destino
        new_rods = [list(peg) for peg in self.rods] # Crea una copia de las varillas actuales para no modificar el estado original
        disk = new_rods[from_peg].pop() # Saca el disco superior de la varilla de origen
        new_rods[to_peg].append(disk) # Coloca el disco en la varilla de destino
        return HanoiState(new_rods, self.num_disks) # Devuelve un nuevo estado con las varillas actualizadas.

    def __hash__(self): # Devuelve un valor hash del estado, lo que permite usarlo en estructuras de datos como conjuntos y diccionarios.
        return hash(self.rods)

    def __eq__(self, other): # Compara dos estados para ver si son iguales, es decir, si las varillas tienen la misma configuración de discos.
        return self.rods == other.rods

    def __repr__(self): # Devuelve una representación en cadena del estado, mostrando los discos en cada varilla.
        return f"A:{list(self.rods[0])} | B:{list(self.rods[1])} | C:{list(self.rods[2])}"

In [ ]:
def heuristica_propia(state: HanoiState, target_peg=2) -> float: # Calcula una heurística admisible y consistente para el problema de las Torres de Hanoi.
    h = 0.0
    base_correcta = True
    
    for disk in range(state.num_disks, 0, -1): # Itera sobre los discos desde el más grande hasta el más pequeño
        rod_actual = None # Inicializa la varilla actual del disco como None
        pos_actual = None # Inicializa la posición actual del disco como None
        
        for r in range(3): # Itera sobre las varillas para encontrar la varilla y posición del disco actual
            if disk in state.rods[r]: # Si el disco está en la varilla r
                rod_actual = r # Guarda la varilla actual del disco
                pos_actual = state.rods[r].index(disk) # Guarda la posición del disco en la varilla
                break

        if rod_actual == target_peg and base_correcta: # Si el disco ya está en la varilla objetivo y la base es correcta, no se necesita moverlo, así que se continúa con el siguiente disco.
            continue
        else:
            base_correcta = False # Si encontramos un disco que no está en la varilla objetivo, marcamos que la base ya no es correcta.
            h += 2 ** (disk - 1) # Agrega el costo exponencial de mover la sub-torre de este disco a la heurística.
            
            h += len(state.rods[rod_actual]) - 1 - pos_actual # Penaliza por los discos que están encima del disco actual en su varilla actual.
            
            if rod_actual != target_peg: # Si el disco no está en la varilla objetivo, penaliza por los discos que están en la varilla objetivo que bloquean la llegada del disco actual.
                h += len(state.rods[target_peg]) # Penaliza por los discos que están en la varilla objetivo que bloquean la llegada del disco actual.

    return float(h)

In [ ]:
class NodeAStar: # Representa un nodo en el árbol de búsqueda A*.
    def __init__(self, state, parent=None, action=None, g=0.0, h=0.0): # Inicializa un nodo con su estado, nodo padre, acción que llevó a este estado, costo acumulado g y heurística h.
        self.state = state # Guarda el estado actual del nodo
        self.parent = parent # Guarda el nodo padre del nodo actual
        self.action = action # Guarda la acción que llevó al nodo actual desde el nodo padre
        self.g = g # Costo acumulado desde el nodo inicial hasta el nodo actual
        self.h = h # Heurística estimada desde el nodo actual hasta el nodo objetivo
        self.f = g + h # Costo total estimado (f = g + h) para el nodo actual

    def __lt__(self, other): # Compara dos nodos según su valor f (costo total estimado) para determinar cuál tiene mayor prioridad en la cola de prioridad.
        return self.f < other.f # Devuelve True si el nodo actual tiene un valor f menor que el nodo otro.

In [ ]:
def busqueda_a_estrella(initial_state: HanoiState, target_peg=2): # Implementa el algoritmo de búsqueda A* para encontrar la secuencia de acciones que lleva al estado objetivo.
    start_node = NodeAStar( 
        state=initial_state,
        parent=None,
        action=None,
        g=0.0,
        h=heuristica_propia(initial_state, target_peg)
    ) # Crea el nodo inicial con el estado inicial, sin padre ni acción, costo g=0 y heurística calculada.
    
    frontier = [] # Cola de prioridad para los nodos a explorar, ordenada por el valor f (costo total estimado)
    heapq.heappush(frontier, start_node) # Agrega el nodo inicial a la cola de prioridad
    
    explored = {initial_state: 0.0} # Diccionario para rastrear los estados explorados y su costo g asociado
    nodos_expandidos = 0 # Contador de nodos expandidos durante la búsqueda

    while frontier: # Mientras haya nodos en la cola de prioridad
        current_node = heapq.heappop(frontier) # Extrae el nodo con el menor valor f de la cola de prioridad
        nodos_expandidos += 1 # Incrementa el contador de nodos expandidos

        if current_node.state.is_goal(target_peg): # Si el estado del nodo actual es un estado objetivo, se ha encontrado la solución
            path = [] # Lista para almacenar la secuencia de acciones que lleva al estado objetivo
            curr = current_node # Comienza desde el nodo actual y retrocede hasta el nodo inicial para reconstruir la ruta
            while curr.action is not None: # Mientras haya una acción que llevó al nodo actual desde su nodo padre
                path.append(curr.action) # Agrega la acción a la lista de ruta
                curr = curr.parent # Mueve al nodo padre para continuar reconstruyendo la ruta
            path.reverse() # Invierte la lista de ruta para que esté en el orden correcto desde el nodo inicial hasta el nodo objetivo
            return path, current_node.g, nodos_expandidos # Devuelve la secuencia de acciones, el costo total g del nodo objetivo y el número de nodos expandidos

        for action in current_node.state.get_valid_actions(): # Itera sobre todas las acciones válidas desde el estado del nodo actual
            child_state = current_node.state.result(action) # Genera el estado hijo resultante de aplicar la acción al estado del nodo actual
            new_g = current_node.g + 1.0 # Calcula el nuevo costo g para el estado hijo, que es el costo del nodo actual más 1 (costo de la acción)
            
            if child_state not in explored or new_g < explored[child_state]: # Si el estado hijo no ha sido explorado o si se encuentra un camino más corto hacia él
                explored[child_state] = new_g # Actualiza el costo g del estado hijo en el diccionario de estados explorados
                h_val = heuristica_propia(child_state, target_peg) # Calcula la heurística para el estado hijo
                child_node = NodeAStar(
                    state=child_state,
                    parent=current_node,
                    action=action,
                    g=new_g,
                    h=h_val
                ) # Crea un nuevo nodo hijo con el estado hijo, el nodo actual como padre, la acción que llevó al estado hijo, el nuevo costo g y la heurística calculada
                heapq.heappush(frontier, child_node) # Agrega el nodo hijo a la cola de prioridad para su exploración futura

    return None, float('inf'), nodos_expandidos # Si la cola de prioridad se vacía sin encontrar un estado objetivo, devuelve None para la ruta, infinito para el costo y el número de nodos expandidos.

In [ ]:
if __name__ == "__main__": # Punto de entrada del programa, ejecuta la búsqueda A* para el problema de las Torres de Hanoi con un número específico de discos.
    NUM_DISCOS = 5
    estado_inicial = HanoiState(
        rods=[list(range(NUM_DISCOS, 0, -1)), [], []],
        num_disks=NUM_DISCOS
    ) # Estado inicial: Varilla 0 con todos los discos de mayor a menor
    
    acciones, costo_final, nodos = busqueda_a_estrella(estado_inicial, target_peg=2) # Ejecuta la búsqueda A* desde el estado inicial hasta la varilla objetivo (por defecto, la varilla 2) y obtiene la secuencia de acciones, el costo total y el número de nodos expandidos.
    
    nombres_varillas = {0: 'A', 1: 'B', 2: 'C'} #
    print(f"=== Búsqueda A* Propia (Torre de Hanói con {NUM_DISCOS} discos) ===")
    print(f"Costo Total g(n): {int(costo_final)} pasos (Óptimo matemático: {2**NUM_DISCOS - 1})")
    print(f"Nodos Expandidos: {nodos}\n")
    
    print("Secuencia de pasos:")
    for i, (origen, destino) in enumerate(acciones, 1): # Itera sobre la secuencia de acciones obtenida, enumerando cada paso y desempaquetando la varilla de origen y destino de cada acción.
        print(f"  Paso {i:02d}: Mover disco de {nombres_varillas[origen]} -> {nombres_varillas[destino]}")

=== Búsqueda A* Propia (Torre de Hanói con 5 discos) ===
Costo Total g(n): 31 pasos (Óptimo matemático: 31)
Nodos Expandidos: 106

Secuencia de pasos:
  Paso 01: Mover disco de A -> C
  Paso 02: Mover disco de A -> B
  Paso 03: Mover disco de C -> B
  Paso 04: Mover disco de A -> C
  Paso 05: Mover disco de B -> A
  Paso 06: Mover disco de B -> C
  Paso 07: Mover disco de A -> C
  Paso 08: Mover disco de A -> B
  Paso 09: Mover disco de C -> B
  Paso 10: Mover disco de C -> A
  Paso 11: Mover disco de B -> A
  Paso 12: Mover disco de C -> B
  Paso 13: Mover disco de A -> C
  Paso 14: Mover disco de A -> B
  Paso 15: Mover disco de C -> B
  Paso 16: Mover disco de A -> C
  Paso 17: Mover disco de B -> A
  Paso 18: Mover disco de B -> C
  Paso 19: Mover disco de A -> C
  Paso 20: Mover disco de B -> A
  Paso 21: Mover disco de C -> B
  Paso 22: Mover disco de C -> A
  Paso 23: Mover disco de B -> A
  Paso 24: Mover disco de B -> C
  Paso 25: Mover disco de A -> C
  Paso 26: Mover disco d